In [1]:
import os
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from pathlib import Path

In [2]:
# Cargar variables de entorno desde el archivo .env
ruta_env = Path("../.env").resolve()
load_dotenv(dotenv_path=ruta_env)

# Stage
cadena_stage = quote_plus(
    f"DRIVER={{{os.getenv('SQLSERVER_DRIVER')}}};"
    f"SERVER={os.getenv('SQLSERVER_SERVIDOR')};"
    f"DATABASE={os.getenv('SQLSERVER_BASE_DATOS')};"
    f"UID={os.getenv('SQLSERVER_USUARIO')};"
    f"PWD={os.getenv('SQLSERVER_PASSWORD')};"
    f"TrustServerCertificate=yes;"
)

engine_stage = create_engine(
    f"mssql+pyodbc:///?odbc_connect={cadena_stage}"
)


# DW

cadena_dw = quote_plus(
    f"DRIVER={{{os.getenv('SQLSERVER_DRIVER_DW')}}};"
    f"SERVER={os.getenv('SQLSERVER_SERVIDOR_DW')};"
    f"DATABASE={os.getenv('SQLSERVER_BASE_DATOS_DW')};"
    f"UID={os.getenv('SQLSERVER_USUARIO_DW')};"
    f"PWD={os.getenv('SQLSERVER_PASSWORD_DW')};"
    f"TrustServerCertificate=yes;"
)

engine_dw = create_engine(
    f"mssql+pyodbc:///?odbc_connect={cadena_dw}"
)

In [3]:
# Datos Stage
esquema_stage = os.getenv("SQLSERVER_ESQUEMA")

consulta = f"""
SELECT
    cliente_id,
    nombres,
    apellidos,
    genero,
    ciudad,
    departamento,
    pais,
    fecha_registro,
    activo
FROM {esquema_stage}.Cliente
"""

df_cliente = pd.read_sql(
    consulta,
    engine_stage
)

print("Registros extraídos:", len(df_cliente))



Registros extraídos: 800


In [ ]:
# Transformación de datos

# Eliminar registros sin identificador.
df_cliente = df_cliente.dropna(
    subset=["cliente_id"]
)

# Eliminar posibles duplicados por clave natural.
df_cliente = df_cliente.drop_duplicates(
    subset=["cliente_id"]
)

# Limpiar espacios en nombres y apellidos.
df_cliente["nombres"] = df_cliente["nombres"].str.strip()
df_cliente["apellidos"] = df_cliente["apellidos"].str.strip()

# La clave del sistema origen se conserva con un nombre explícito.
df_cliente = df_cliente.rename(
    columns={
        "cliente_id": "cliente_id_origen"
    }
)

print("Registros transformados:", len(df_cliente))

In [ ]:
# Carga incremental en el Data Warehouse
esquema_dw = os.getenv("SQLSERVER_ESQUEMA_DW")

# Leer las claves de negocio que ya existen en la dimensión.
df_existentes = pd.read_sql(
    f"""
    SELECT cliente_id_origen
    FROM {esquema_dw}.DimCliente
    """,
    engine_dw
)

# Conservar solamente los clientes nuevos.
df_cliente_nuevos = df_cliente[
    ~df_cliente["cliente_id_origen"].isin(
        df_existentes["cliente_id_origen"]
    )
].copy()


In [ ]:
df_cliente_nuevos.head()

In [ ]:

# Insertar únicamente los registros nuevos.
if not df_cliente_nuevos.empty:

    df_cliente_nuevos.to_sql(
        name="DimCliente",
        con=engine_dw,
        schema=esquema_dw,
        if_exists="append",
        index=False
    )

    print(
        f"{len(df_cliente_nuevos)} clientes nuevos cargados en dbo.DimCliente."
    )

else:
    print("No existen clientes nuevos para cargar.")

In [ ]:
# Cerrar conexiones
engine_stage.dispose()
engine_dw.dispose()